<div style="font-size: 0.85em;">

### Few‑Shot Prompting in LangChain

#### What is Few‑Shot Prompting?
Few‑shot prompting means giving the model **a few examples** of the task before asking it to answer a new question.

Each example has:
- **Input** – what the user asked  
- **Output** – the ideal answer

The model learns the pattern from the examples and uses it to answer new questions in the same format.

#### Why Use Few‑Shot Prompting?
- Helps the model produce **consistent** answers.  
- Good for tasks where the **format matters** (e.g., classification, translation, structured extraction).  
- Reduces the need for long instructions.  
- Improves output **reliability** for specific tasks.

### How It Works

[Examples]
Example 1: Input -> Output
Example 2: Input -> Output
Example 3: Input -> Output
|
v
[Prompt Template]

Combines examples + new question
|
v
[Chat Model]
|
v
[Answer]


## Key LangChain Component
`FewShotChatMessagePromptTemplate` is used to build prompts with examples.

We combine it with:
- `ChatPromptTemplate` – the main prompt  
- `ChatOpenAI` – the model  
- `StrOutputParser` – to get the final text

In this notebook, we will:
1. Create examples.  
2. Build a few‑shot prompt.  
3. Send a new question to the model.  
4. See how the model follows the examples.

</div>

In [1]:
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import FewShotChatMessagePromptTemplate, ChatPromptTemplate
from dotenv import load_dotenv

In [2]:
# Load environment variables
load_dotenv()

True

Step 2: Define Examples

In [9]:
# Define a list of examples.
examples = [
    {'input': 'I absolutely loved this movie!', 'output': 'positive'},
    {'input': 'It was boring and too long.', 'output': 'negative'},
    {'input': 'The acting was great but the plot was weak.', 'output': 'mixed'}
]

print('Examples:')
for example in examples:
    print(example)

Examples:
{'input': 'I absolutely loved this movie!', 'output': 'positive'}
{'input': 'It was boring and too long.', 'output': 'negative'}
{'input': 'The acting was great but the plot was weak.', 'output': 'mixed'}


Step 3: Create the Example Prompt

In [10]:
# Create a template for how each example will be formatted.
example_prompt = ChatPromptTemplate.from_messages([
    ('human', '{input}'),
    ('ai', '{output}')
])

print('Example prompt template created:')
print(example_prompt)

Example prompt template created:
input_variables=['input', 'output'] messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['input'], template='{input}')), AIMessagePromptTemplate(prompt=PromptTemplate(input_variables=['output'], template='{output}'))]


Step 4: Format One Example

In [ ]:
# Use the example_prompt to format the first example
formatted_example = example_prompt.format_messages(
    input='I absolutely loved this movie!',
    output='positive'
)

# Print the formatted messages
print('Formatted messages for one example:')
for message in formatted_example:
    print(f'[{message.type}] {message.content}')

Formatted messages for one example:
[human] I absolutely loved this movie!
[ai] positive


Step 5: Create the Few-Shot Prompt Template

In [14]:
# Create the few-shot prompt template.
# It takes the examples and the example_prompt, and knows how to insert them.
few_shot_prompt = FewShotChatMessagePromptTemplate(
    examples=examples,
    example_prompt=example_prompt,
    input_variables=['input']
)

# Format the few-shot prompt to see the messages it generates.
example_messages = few_shot_prompt.format_messages()
print('Messages generated by few_shot_prompt')

for msg in example_messages:
    print(f'[{msg.type}] {msg.content}')


Messages generated by few_shot_prompt
[human] I absolutely loved this movie!
[ai] positive
[human] It was boring and too long.
[ai] negative
[human] The acting was great but the plot was weak.
[ai] mixed


Step 6: Create the Final Prompt

In [16]:
# Create the final prompt template.
# It contains:
# 1. A system message that sets the task.
# 2. The few_shot_prompt object (which expands to all example messages).
# 3. A human message with a placeholder {input} for the new review.

final_prompt = ChatPromptTemplate.from_messages([
    ('system', 'You are a sentiment classifier. Classify the review as positive, negative, or mixed.'),
    few_shot_prompt,
    ('human', '{input}')
])

Step 7: Inspect the Final Formatted Messages

In [17]:
# Format the final prompt with a new review
formatted_messages = final_prompt.format_messages(
    input='The visuals were stunning but the story was confusing.'
)

# Print each message
print('Full formatted messages sent to the model:')
for msg in formatted_messages:
    print(f'[{msg.type}] {msg.content}')
    print('---')

Full formatted messages sent to the model:
[system] You are a sentiment classifier. Classify the review as positive, negative, or mixed.
---
[human] I absolutely loved this movie!
---
[ai] positive
---
[human] It was boring and too long.
---
[ai] negative
---
[human] The acting was great but the plot was weak.
---
[ai] mixed
---
[human] The visuals were stunning but the story was confusing.
---


Step 8: Create the Chat Model

In [18]:
llm = ChatOpenAI(model='gpt-4o-mini', temperature=0)

Build and Run the Chain

In [19]:
chain = final_prompt | llm | StrOutputParser()

# Invoke the chain with a new review
result = chain.invoke(
    {'input': 'The visuals were stunning but the story was confusing.'}
)

print(f'Predicted sentiment: {result}')

Predicted sentiment: mixed


Language Translation

In [29]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate, FewShotChatMessagePromptTemplate
from langchain_core.output_parsers import StrOutputParser
from dotenv import load_dotenv

# Load .env
load_dotenv()

# 2. Define two simple examples (English phrase -> Spanish translation)
examples = [
    {'english': 'Good morning', 'spanish': 'Buenos días'},
    {'english': 'Thank you', 'spanish': 'Gracias'}
]

# 3. Define how each example should look: human says English, AI says Spanish
example_prompt = ChatPromptTemplate.from_messages([
    ('human', '{english}'),
    ('ai', '{spanish}')                      
])

# 4. Create the few-shot prompt using the examples
few_shot_prompt = FewShotChatMessagePromptTemplate(
    examples=examples,
    example_prompt=example_prompt,
    input_variables=['english']
)

# 5. Create the final prompt: system instruction + examples + new phrase
final_prompt = ChatPromptTemplate.from_messages([
    ('system', 'Translate the following English phrases to Spanish.'),
    few_shot_prompt,
    ('human', '{english}')
])

# 6. Create the chat model (deterministic)
llm = ChatOpenAI(model='gpt-4o-mini', temperature=0)

# 7. Build the chain: prompt -> model -> string output
chain = final_prompt | llm | StrOutputParser()

# 8. Run with a new phrase
result = chain.invoke({'english': 'Good night and goodbye ma'})
print(f'Translation: {result}')

Translation: Buenas noches y adiós, mamá.


Translation
* few‑shot prompting example for translating English to Yoruba

In [39]:
# 1. Import 
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate, FewShotChatMessagePromptTemplate
from langchain_core.output_parsers import StrOutputParser
from dotenv import load_dotenv

# Load .env
load_dotenv()

# 2. Define two examples (English phrase -> Yoruba translation)
examples = [
    {'english': 'Good morning', 'yoruba': 'Ẹ káàárọ̀'},
    {'english': 'Thank you', 'yoruba': 'Ẹ ṣé'}
]

# 3. Define how each example should look: human says English, AI says Yoruba
example_prompt = ChatPromptTemplate.from_messages([
    ('human', '{english}'),
    ('ai', '{yoruba}')
])

# 4. Create the few-shot prompt using the examples
#    input_variables must match the variable name used in the final prompt's human message.
few_shot_prompt = FewShotChatMessagePromptTemplate(
    examples=examples,
    example_prompt=example_prompt,
    input_variables=['english']
)

# 5. Create the final prompt: system instruction + examples + new phrase placeholder
final_prompt = ChatPromptTemplate.from_messages([
    ('system', 'Translate the following English phrases to Yoruba.'),
    few_shot_prompt,
    ('human', '{english}')
])

# 6. Create the chat model 
llm = ChatOpenAI(model='gpt-4o-mini', temperature=0)

# 7. Build the chain
chain = final_prompt | llm | StrOutputParser()

# 8. Run with a new English phrase
result = chain.invoke({'english': 'I want to eat rice, meat, fish and plantain, then going to school for mathematics class and finally sleep.'})
print(f'Translation: {result}')

Translation: Mo fẹ́ jẹ́ iyan, ẹran, ẹja àti dodo, lẹ́yìn náà mo fẹ́ lọ sí ilé-ẹ̀kọ́ fún kíláàsì ìṣirò, àti nígbàtí mo parí, mo fẹ́ sùn.
